# Ingesta de datos - Capa Bronze

Este notebook realiza la ingesta de archivos CSV del Banco Mundial hacia la capa **Bronze** del Data Lake, sin aplicar transformaciones.

In [7]:
from pathlib import Path
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze-worldbank-ingestion")
    .getOrCreate()
)

current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
RAW_ROOT = PROJECT_ROOT / "raw"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"

BRONZE_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw root: {RAW_ROOT}")
print(f"Bronze root: {BRONZE_ROOT}")

Project root: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation
Raw root: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\raw
Bronze root: c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze


In [14]:
import csv
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

INDICATOR_FOLDERS = {
    "pib_per_capita": "PIB per cápita (US$ a precios actuales)",
    "indice_gini": "Índice de Gini",
    "desempleo": "Desempleo, total (% de la fuerza laboral total) (estimación modelada de la OIT)",
    "inflacion": "Inflación, deflactor del PIB (%) anual",
    "exportaciones": "Exportaciones de bienes y servicios (% del PIB)",
    "importaciones": "Importaciones de bienes y servicios (% del PIB)",
    "poblacion": "Población, total",
}


def read_world_bank_csv(source_csv_path: Path):
    # Leer el archivo con Python para obtener el header real (línea 4)
    with open(source_csv_path, 'r', encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        for _ in range(4):
            next(reader)
        header = next(reader)
    
    # Crear schema con StringType para todas las columnas
    schema = StructType([
        StructField(col_name.strip(), StringType(), True)
        for col_name in header
    ])
    
    # Leer el CSV con el schema correcto, saltando las primeras 5 líneas
    df = (
        spark.read
        .option("header", False)
        .option("encoding", "UTF-8")
        .schema(schema)
        .csv(str(source_csv_path))
    )
    
    # Filtrar solo las filas de datos (excluir metadata)
    first_col = header[0]
    return df.filter(
        (~F.col(first_col).isin("Data Source", "Last Updated Date")) &
        (F.col(first_col).isNotNull()) &
        (F.trim(F.col(first_col)) != "")
    )

In [15]:
for indicator_key, folder_name in INDICATOR_FOLDERS.items():
    source_folder = RAW_ROOT / folder_name

    if not source_folder.exists():
        raise FileNotFoundError(f"No existe la carpeta origen para '{indicator_key}': {source_folder}")

    csv_files = sorted(source_folder.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No se encontró CSV en la carpeta: {source_folder}")

    source_csv = csv_files[0]
    target_path = BRONZE_ROOT / indicator_key

    df_raw = read_world_bank_csv(source_csv)

    (
        df_raw.write
        .mode("overwrite")
        .parquet(str(target_path))
    )

    print(f"[OK] {indicator_key} -> {target_path}")

print("Ingesta Bronze completada.")
    

[OK] pib_per_capita -> c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze\pib_per_capita
[OK] indice_gini -> c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze\indice_gini
[OK] desempleo -> c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze\desempleo
[OK] inflacion -> c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze\inflacion
[OK] exportaciones -> c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze\exportaciones
[OK] importaciones -> c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze\importaciones
[OK] poblacion -> c:\Users\Darwin Lenis\OneDrive\Escritorio\Mentoria Data\data-lake-simulation\data\bronze\poblacion
Ingesta Bronze completada.
